In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embedding_dim: int, attention_dim: int, num_heads: int):
        super().__init__()
        torch.manual_seed(0)
        assert attention_dim % num_heads == 0

        self.C = embedding_dim
        self.h = num_heads
        self.hd_k = attention_dim
        self.d_k = attention_dim // num_heads # head dimension

        self.W_K = nn.Linear(self.C, self.hd_k, bias = False)
        self.W_Q = nn.Linear(self.C, self.hd_k, bias = False)
        self.W_V = nn.Linear(self.C, self.hd_k, bias = False)

        self.W_O = nn.Linear(self.hd_k, self.C, bias = False)

    def forward(self, embedded):
        B,T,C = embedded.shape
        h, d_k, hd_k = self.h, self.d_k, self.hd_k

        # 1. Projection
        K = self.W_K(embedded)      # [B, T, hd_k]   aka   [B,T,A]
        Q = self.W_Q(embedded)      # [B, T, hd_k]   aka   [B,T,A]
        V = self.W_V(embedded)      # [B, T, hd_k]   aka   [B,T,A]

        # 1b. Split into heads
        K = K.view(B, T, h, d_k).transpose(1,2)     # [B, h, T, d_k]
        Q = Q.view(B, T, h, d_k).transpose(1,2)     # [B, h, T, d_k]
        V = V.view(B, T, h, d_k).transpose(1,2)     # [B, h, T, d_k]

        # 2. Attention Scores
        attention_matrix = Q @ K.transpose(-2, -1)  # [B, h, T, T]
        attention_scores = attention_matrix // (d_k ** 0.5)

        # 3. Causal Mask
        mask = torch.tril(torch.ones(T, T)) # [T, T]
        mask = (mask == 0)
        masked_scores = attention_scores.masked_fill(mask, float('-inf'))

        # 4. Softmax
        A = F.softmax(masked_scores, dim = -1) # [B, h , T, T] (no change)

        # 5. Blending Values
        Z = A @ V               #[B, h, T, d_k]

        # 6. Merge outputs
        Z = Z.transpose(1, 2).contiguous().view(B, T, hd_k) # We need to make # Example: T = 5 ("The tall man runs ."), h = 2, d_k = 2
        #   A @ V             [1, 2, 5, 2]  strides (20, 10, 2, 1)
        #   .transpose(1, 2)  [1, 5, 2, 2]  strides (20, 2, 10, 1)   no copy, only strides swap
        # view must merge h and d_k,
        # which needs stride[h] == stride[d_k] * size[d_k]:
        #   10 != 1 * 2  -> view raises an error
        # .contiguous() copies into (20, 4, 2, 1): 2 == 1 * 2 -> view works -> [1, 5, 4], strides (20, 4, 1)
        # (.reshape would do the same copy silently)


        # 7. Output Projection
        out = self.W_O(Z)
        return out

In [ ]:
# Just for shits and giggles, I am writing the Single Head Attention code here as well.

In [ ]:
class SingleHeadSelfAttention(nn.Module):

    def __init__(self, embedding_dim: int, attention_dim: int):
        super().__init__()
        torch.manual_seed(0)

        self.C = embedding_dim
        self.d_k = attention_dim

        self.W_K = nn.Linear(self.C, self.d_k, bias = False)
        self.W_Q = nn.Linear(self.C, self.d_k, bias = False)
        self.W_V = nn.Linear(self.C, self.d_k, bias = False)


    def forward(self, embedded):
        B,T,C = embedded.shape
        d_k = self.d_k

        # 1. Projection
        K = self.W_K(embedded)      # [B, T, d_k]   aka   [B,T,A]
        Q = self.W_Q(embedded)      # [B, T, d_k]   aka   [B,T,A]
        V = self.W_V(embedded)      # [B, T, d_k]   aka   [B,T,A]

        # 1b. Split into heads - (none)

        # 2. Attention Scores
        attention_matrix = Q @ K.transpose(-2, -1)  # [B, T, T]
        attention_scores = attention_matrix // (d_k ** 0.5)

        # 3. Causal Mask
        mask = torch.tril(torch.ones(T, T)) # [T, T]
        mask = (mask == 0)
        masked_scores = attention_scores.masked_fill(mask, float('-inf'))

        # 4. Softmax
        A = F.softmax(masked_scores, dim = -1) # [B, T, T] (no change)

        # 5. Blending Values
        Z = A @ V               #[B, T, d_k]

        # 6. Merge outputs - (none)
        # 7. Output Projection - (none)

        return Z